In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import missingno as msno
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
import os

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
# from catboost import CatBoostClassifier
from sklearn import metrics
from sklearn.metrics import roc_curve
from sklearn.metrics import recall_score, confusion_matrix, precision_score, f1_score, accuracy_score, classification_report

In [ ]:
df = pd.read_csv('../data/customer churn data.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
#Checking missing values
df.isnull().sum()

In [ ]:
df.drop(['customerID'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df.isnull().sum()

In [ ]:
df[df['TotalCharges'].isnull()]

In [ ]:
df[df['tenure']==0]

In [ ]:
df.drop(df[df['tenure']==0].index, axis=0, inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df['SeniorCitizen'] = df['SeniorCitizen'].replace({1: 'Yes', 0:'No'})

In [ ]:
df.head()

In [ ]:
df['InternetService'].describe()

In [ ]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

df[numerical_cols].describe()

In [ ]:
gender_labels = ['Male', 'Female']
citizen_labels = ['No', 'Yes']

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'domain'}, {'type':'domain'}]])
fig.add_trace(go.Pie(labels=gender_labels, values=df['gender'].value_counts(), name="Gender"), 1, 1)
fig.add_trace(go.Pie(labels=citizen_labels, values=df['Churn'].value_counts(), name="Churn"), 1, 2)

fig.update_traces(hole=.4, hoverinfo="label+percent+name", textfont_size=16)

fig.update_layout(
    title_text="Gender and Churn Distributions",
    # Add annotations in the center of the donut pies.
    annotations=[dict(text='Gender', x=0.16, y=0.5, font_size=20, showarrow=False),
                 dict(text='Churn', x=0.84, y=0.5, font_size=20, showarrow=False)])

fig.show()

In [ ]:
df['Churn'][df['Churn']=='No'].groupby(df['gender']).count()

In [ ]:
df['Churn'][df['Churn']=='Yes'].groupby(df['gender']).count()

In [ ]:
fig = px.histogram(df, x='Churn', color='Contract', barmode='group', title='Customer contract distribution')
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

In [ ]:
labels = df['PaymentMethod'].unique()
values = df['PaymentMethod'].value_counts()

fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3)])
fig.update_layout(title_text='Payment Method Distribution')
fig.show()

In [ ]:
fig = px.histogram(df, x='Churn', color='PaymentMethod', title="Customer Payment Method Distribution")

fig.show()

In [ ]:
df.groupby(['InternetService', 'Churn'])['gender'].value_counts()

In [ ]:
df.groupby(['Churn'])['InternetService'].value_counts()

In [ ]:
import plotly.graph_objects as go

# 1. Prepare data into a structured pivot table
# Rows: (Churn, gender) MultiIndex | Columns: InternetService
counts = (
    df.groupby(['Churn', 'gender', 'InternetService'])
    .size()
    .unstack(level='InternetService', fill_value=0)
)

# 2. Build multi-category x-axis coordinates directly from the index
x_coords = [
    [f"Churn: {c}" for c in counts.index.get_level_values('Churn')],
    list(counts.index.get_level_values('gender'))
]

# 3. Add each InternetService trace dynamically
fig = go.Figure()
for service in counts.columns:
    fig.add_trace(go.Bar(
        x=x_coords,
        y=counts[service],
        name=service,
        text=counts[service],
        textposition='auto'
    ))

# 4. Polish layout
fig.update_layout(
    title_text="<b>Churn Distribution w.r.t. Internet Service and Gender</b>",
    barmode='group',
    xaxis_title="Churn Status & Gender",
    yaxis_title="Customer Count",
    template="plotly_white",
    legend_title_text="Internet Service",
    bargap=0.2,
    bargroupgap=0.05
)

fig.show()

In [ ]:
fig = px.histogram(df, x='Churn', color='Dependents', barmode='group', title='Dependents Distribution')
fig.show()

In [ ]:
px.histogram(df, x='Churn', color='Partner', barmode='group', title='Churn Distribution according to relationship status')

In [ ]:
px.histogram(df, x='Churn', color='SeniorCitizen', barmode='group', title='Churn distribution of citizen status')

In [ ]:

fig = px.histogram(df, x="Churn", color="OnlineSecurity", barmode="group", title="<b>Churn w.r.t Online Security</b>")
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

In [ ]:
fig = px.histogram(df, x="Churn", color="PaperlessBilling",  title="<b>Churn distribution w.r.t. Paperless Billing</b>")
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

In [ ]:
fig = px.histogram(df, x="Churn", color="TechSupport",barmode="group",  title="<b>Churn distribution w.r.t. TechSupport</b>")
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

In [ ]:
fig = px.histogram(df, x="Churn", color="PhoneService", title="<b>Churn distribution w.r.t. Phone Service</b>")
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

In [ ]:
ax = sns.kdeplot(df.TotalCharges[(df["Churn"] == 'No') ],
                color="Gold", shade = True);
ax = sns.kdeplot(df.TotalCharges[(df["Churn"] == 'Yes') ],
                ax =ax, color="Green", shade= True);
ax.legend(["Not Churn","Churn"],loc='upper right');
ax.set_ylabel('Density');
ax.set_xlabel('Total Charges');
ax.set_title('Distribution of total charges by churn');

In [ ]:
fig = px.box(df, x='Churn', y = 'tenure')

# Update yaxis properties
fig.update_yaxes(title_text='Tenure (Months)', row=1, col=1)
# Update xaxis properties
fig.update_xaxes(title_text='Churn', row=1, col=1)

# Update size and title
fig.update_layout(autosize=True, width=750, height=600,
    title_font=dict(size=25, family='Courier'),
    title='<b>Tenure vs Churn</b>',
)

fig.show()

In [ ]:
plt.figure(figsize=(25, 10))

corr = df.apply(lambda x: pd.factorize(x)[0]).corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

ax = sns.heatmap(corr, mask=mask, xticklabels=corr.columns, yticklabels=corr.columns, annot=True, linewidths=.2, cmap='coolwarm', vmin=-1, vmax=1)

In [ ]:
df

In [ ]:
def binary_formatting(dataframe):
    features = dataframe.columns
    for feature in features: 
        if set(dataframe[feature].unique()) == {'Yes', 'No'}:
            dataframe[feature] = (dataframe[feature]=='Yes').astype('int')

for col in df.columns:
    binary_formatting(df)


In [ ]:
categorical_columns = df.select_dtypes(include=['object']).columns

for col in categorical_columns:
    print(df[col].unique())

In [ ]:
df['Contract'] = df['Contract'].replace({'Month-to-month':0, 'One year':1, 'Two year':2})

In [ ]:
oh_encoder = OneHotEncoder(sparse_output=False, drop='first').set_output(transform='pandas')

encoded_df = oh_encoder.fit_transform(df[['gender',
'MultipleLines',
'InternetService',
'OnlineSecurity',
'OnlineBackup',
'DeviceProtection',
'TechSupport',
'StreamingTV',
'StreamingMovies',
'PaymentMethod']])

In [ ]:
import pickle

In [ ]:


models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

encoder_path = os.path.join(models_dir, 'oh_encoder.pkl')


with open(encoder_path, 'wb') as f:
    pickle.dump(oh_encoder, f)

print(f'encoder sucessful dumped to {encoder_path}')

In [ ]:
encoded_df

In [ ]:
new_df = pd.concat([df, encoded_df], axis=1)

In [ ]:
new_df.drop(columns=['gender',
'MultipleLines',
'InternetService',
'OnlineSecurity',
'OnlineBackup',
'DeviceProtection',
'TechSupport',
'StreamingTV',
'StreamingMovies',
'PaymentMethod'], axis=1, inplace=True)

In [ ]:
new_df.corr()['Churn'].sort_values(ascending=False)

In [ ]:
new_df['num_services'] = new_df[['PhoneService','OnlineSecurity_Yes','OnlineBackup_Yes',
    'DeviceProtection_Yes','TechSupport_Yes','StreamingTV_Yes','StreamingMovies_Yes']].sum(axis=1)
new_df['avg_monthly_spend'] = new_df['TotalCharges'] / new_df['tenure']
new_df['charge_gap'] = new_df['MonthlyCharges'] - new_df['avg_monthly_spend']  # recent price hike?
new_df['is_new_customer'] = (new_df['tenure'] <= 6).astype(int)
new_df['autopay'] = 1 - new_df['PaymentMethod_Electronic check'] - new_df['PaymentMethod_Mailed check']

In [ ]:
X = new_df.drop(columns=['Churn'])
y = new_df['Churn']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=40)

In [ ]:
def distplot(feature, dataframe):
    plt.figure(figsize=(8,3))
    plt.title(f'Distribution of {feature}')
    ax = sns.histplot(dataframe[feature])


In [ ]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

for feat in numerical_cols:
    distplot(feat, df)

In [ ]:
scaled_df = pd.DataFrame(StandardScaler().fit_transform(df[numerical_cols].astype('float64')), columns=numerical_cols)

In [ ]:
for col in numerical_cols:
    distplot(col, scaled_df)

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.utils import shuffle
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline as imbPipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

smote = SMOTE(random_state=42)
# X_train_sampled, y_train_sampled = smote.fit_resample(X_train, y_train)
# X_train_sampled, y_train_sampled = shuffle(
#     X_train_sampled, y_train_sampled, random_state=42)

In [ ]:
scaler = StandardScaler()



X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


scaler_path = os.path.join(models_dir, 'scaler.pkl')

with open (scaler_path, 'wb') as f:
    pickle.dump(scaler, f)



In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
models = {'random forest': RandomForestClassifier(random_state=42, n_estimators=200, max_depth=8, min_samples_leaf=5),
          'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6, min_samples_leaf=10),
          'xgboost': XGBClassifier(random_state=42, learning_rate=0.05, eval_metric='logloss', n_estimators=150)}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {}

for model_name, model in models.items():
    pipe = imbPipeline([('smote', SMOTE(random_state=42)), ('model', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    cv_scores[model_name] = scores
    print(f"{model_name}'s cross value f1 score: {np.mean(scores):.3f}")





#### Predicting with best model

In [ ]:
best_pipe = imbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', models['xgboost'])
])
best_pipe.fit(X_train, y_train)

y_pred = best_pipe.predict(X_test)
print("\n=== Final Test Set Evaluation (XGBoost) ===")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
bestrf_pipe = imbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', models['random forest'])
])
bestrf_pipe.fit(X_train, y_train)

y_pred = bestrf_pipe.predict(X_test)
print("\n=== Final Test Set Evaluation (random forest) ===")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
pipeline = imbPipeline([('smote', SMOTE(random_state=42)),
                        ('model', RandomForestClassifier(random_state=42))])

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 8, 10, None],
    'model__min_samples_split': [2,5,10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(estimator=pipeline,
                          param_grid=param_grid,
                          cv=cv,
                          scoring='f1')
grid_search.fit(X_train, y_train)

print(f'Best f1 score: {grid_search.best_score_:.3f}')
print(f'Best Paramaters: {grid_search.best_params_}')

In [ ]:

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'model__n_estimators': [200, 400, 600],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__max_depth': [3, 4, 5, 6],
    'model__min_child_weight': [1, 3, 5, 10],
    'model__subsample': [0.7, 0.85, 1.0],
    'model__colsample_bytree': [0.6, 0.8, 1.0],
    'model__reg_lambda': [1, 5, 10],
}

pipe = imbPipeline([('smote', SMOTE(random_state=42)),
                    ('model', XGBClassifier(random_state=42, eval_metric='logloss'))])

search = RandomizedSearchCV(pipe, param_dist, n_iter=40, scoring='roc_auc',
                            cv=cv, n_jobs=-1, random_state=42)
search.fit(X_train, y_train)
print(search.best_params_, search.best_score_)



### AUC-ROC

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score  

In [ ]:
for model_name, model in models.items():
    pipe = imbPipeline([('smote', SMOTE(random_state=42)),
                        ('model', model)])
    pipe.fit(X_train, y_train)

    y_probs = pipe.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_probs)
    auc_score = roc_auc_score(y_test, y_probs)

    plt.plot(fpr, tpr, lw=2, label=f'{model_name} (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Guess (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
plt.ylabel('True Positive Rate (Recall)', fontsize=11)
plt.title('ROC-AUC Curve Comparison on Test Set', fontsize=14, weight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

model_path = os.path.join(models_dir, 'customer churn prediction rf model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(bestrf_pipe, f)